#### 2_prepare_cimmyt_geometries.ipynb

**Author:** James Sayre  
**Email:** jsayre@ucdavis.edu  
**Date Modified:** 2026-02-28

**Description:** Prepares CIMMYT plot geometries for satellite feature extraction in Google Earth Engine. Buffers GPS point locations into circular polygons using farmer-reported plot area, spatial-joins with ADC polygons, and extracts planting months.

**Inputs:**
- `Intermediates/CIMMYT/cimmyt_plot_locations.shp` — Point locations for ~51k CIMMYT plots
- `Intermediates/CIMMYT/cimmyt_maize_yields.csv` — Maize yields by plot-year (filter to 2017-2022)
- `Data/CIMMYT/Farmer_plots/1.-Farmer_Plot_Logbook_2012-2022_03.xlsx` — Plot areas
- `Data/CIMMYT/Farmer_plots/2.-Sowing_harvest_yields_2012-2022_02.xlsx` — Sowing dates
- `Data/planting_months_harmonic_regression.csv` — Fallback phenology by municipality
- `Maize_prediction/Data/Shapefiles/adc_shapefile.shp` — ADC polygons for spatial join

**Outputs:**
- `Data/CIMMYT/cimmyt_plot_geometries_for_ee.csv` — Plot geometries ready for GEE extraction

In [1]:
import os
import numpy  as np
import pandas as pd
import geopandas as gpd
from   shapely import to_geojson, force_2d
from   datetime import datetime

# ── Directories ──────────────────────────────────────────
home_dir    =  os.path.expanduser("~")
proj_dir    =  os.path.join(home_dir, "Dropbox", "Projects", "Maize_prediction")
data_dir    =  os.path.join(proj_dir, "Data")
int_dir     =  os.path.join(proj_dir, "Intermediates", "CIMMYT")
cimmyt_dir  =  os.path.join(data_dir, "CIMMYT", "Farmer_plots")
avo_dir     =  os.path.join(home_dir, "Dropbox", "Projects", "Avocado_Deforestation")

# ── Inputs ───────────────────────────────────────────────
shp_f       =  os.path.join(int_dir, "cimmyt_plot_locations.shp")         # point locations
yields_f    =  os.path.join(int_dir, "cimmyt_maize_yields.csv")           # maize yields
logbook_f   =  os.path.join(cimmyt_dir, "1.-Farmer_Plot_Logbook_2012-2022_03.xlsx")  # plot areas
sowing_f    =  os.path.join(cimmyt_dir, "2.-Sowing_harvest_yields_2012-2022_02.xlsx") # sowing dates
phenology_f =  os.path.join(data_dir, "planting_months_harmonic_regression.csv")      # fallback
adc_shp_f   =  os.path.join(home_dir, "Dropbox", "Projects", "Maize_prediction", "Data", "Shapefiles", "adc_shapefile.shp")  # ADC polygons

# ── Outputs ──────────────────────────────────────────────
out_dir     =  os.path.join(data_dir, "CIMMYT")
os.makedirs(out_dir, exist_ok=True)
out_f       =  os.path.join(out_dir, "cimmyt_plot_geometries_for_ee.csv") # geometry CSV

#### Load shapefile and filter to plots with maize yields in 2017-2022

In [2]:
# Load plot point locations
plots =  gpd.read_file(shp_f)
print(f"Plot locations: {len(plots):,}")

# Load yields, filter to 2017-2022, get unique plot IDs
yields     =  pd.read_csv(yields_f)
yields     =  yields[yields['year'].between(2017, 2022)]
plot_ids   =  yields['plot_id'].astype(str).unique()
print(f"Unique plots with maize yields 2017-2022: {len(plot_ids):,}")

# Filter shapefile to those plots
plots['plot_id'] =  plots['plot_id'].astype(str)
plots =  plots[plots['plot_id'].isin(plot_ids)].copy().reset_index(drop=True)
print(f"Plots with GPS + yields in 2017-2022: {len(plots):,}")

Plot locations: 51,000
Unique plots with maize yields 2017-2022: 18,652
Plots with GPS + yields in 2017-2022: 18,582


#### Extract plot areas from logbook and compute buffer radii

In [3]:
# Load logbook for plot areas
logbook =  pd.read_excel(logbook_f, sheet_name='Data', header=1, engine='openpyxl')

# Get area per plot — use survey-reported area column
area_col =  'TOTAL PLOT AREA (HA) DATA OBTAINED BY SURVEY WITH THE FARMER'
areas =  (logbook[['PLOT ID', area_col]]
          .dropna(subset=[area_col])
          .drop_duplicates(subset='PLOT ID')
          .rename(columns={'PLOT ID': 'plot_id', area_col: 'area_ha'}))
areas['plot_id'] =  areas['plot_id'].astype(str)

# Merge areas onto plots
plots =  plots.merge(areas, on='plot_id', how='left')

# Fill missing areas with median
median_area =  plots['area_ha'].median()
n_missing   =  plots['area_ha'].isna().sum()
plots['area_ha'] =  plots['area_ha'].fillna(median_area)
print(f"Area: median = {median_area:.2f} ha, filled {n_missing} missing values")

# Compute buffer radius in meters: r = sqrt(area_ha * 10000 / pi)
plots['radius_m'] =  np.sqrt(plots['area_ha'] * 10000 / np.pi)
print(f"Buffer radius: median = {plots['radius_m'].median():.1f} m")

Area: median = 1.60 ha, filled 73 missing values
Buffer radius: median = 71.4 m


#### Spatial join with ADC polygons

In [4]:
# Load ADC polygons and spatial join to get adcid for each plot
adcs =  gpd.read_file(adc_shp_f)
adcs =  adcs[['adcid', 'geometry']].dropna(subset=['adcid'])
adcs =  adcs.to_crs(plots.crs)
print(f"ADC polygons: {len(adcs):,}")

# Spatial join (points within ADC polygons)
plots =  gpd.sjoin(plots, adcs, how='left', predicate='within')
plots =  plots.drop(columns=['index_right'])

n_no_adc =  plots['adcid'].isna().sum()
print(f"Plots without ADC match: {n_no_adc}")
print(f"Plots with ADC match: {plots['adcid'].notna().sum():,}")

ADC polygons: 295,184


Plots without ADC match: 576
Plots with ADC match: 18,006


#### Buffer points into circular polygons

In [5]:
# Reproject to EPSG:6372 (Mexico meters), buffer, reproject back
plots =  plots.to_crs("EPSG:6372")
plots['geometry'] =  plots.apply(lambda row: row.geometry.buffer(row['radius_m']), axis=1)
plots =  plots.to_crs("EPSG:4326")
print(f"Buffered {len(plots):,} plots")

Buffered 18,582 plots


#### Extract sowing months from raw yields Excel

In [6]:
# Load sowing dates from raw yields file
sowing =  pd.read_excel(sowing_f, sheet_name='Data', engine='openpyxl')
sowing.columns =  sowing.columns.str.replace('\xa0', ' ')

# Parse sowing dates (two-pass: %Y-%m-%d then %d/%m/%Y)
def parse_sowing_date(val):
    if pd.isna(val):
        return None
    val =  str(val).strip()
    for fmt in ['%Y-%m-%d', '%d/%m/%Y', '%Y-%m-%d %H:%M:%S']:
        try:
            return datetime.strptime(val, fmt)
        except ValueError:
            continue
    # Try pandas as fallback
    try:
        return pd.to_datetime(val)
    except Exception:
        return None

sowing['sow_date']  =  sowing['CROP.SOWING.DATE'].apply(parse_sowing_date)
sowing['sow_month'] =  sowing['sow_date'].apply(lambda x: x.month if x is not None else None)
sowing['plot_id']   =  sowing['PLOT.ID'].astype(str)

# Compute modal sowing month per plot
sow_months =  (sowing.dropna(subset=['sow_month'])
               .groupby('plot_id')['sow_month']
               .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else None)
               .reset_index()
               .rename(columns={'sow_month': 'planting_month'}))

print(f"Plots with parseable sowing dates: {len(sow_months):,}")

Plots with parseable sowing dates: 51,072


In [7]:
# Merge sowing months onto plots
plots =  plots.merge(sow_months, on='plot_id', how='left')

n_missing_month =  plots['planting_month'].isna().sum()
print(f"Plots missing planting month: {n_missing_month}")

# Build muncode (needed for phenology fallback and output)
plots['muncode'] =  (plots['id_state'].fillna(0).astype(int).astype(str).str.zfill(2) +
                      plots['id_mun'].fillna(0).astype(int).astype(str).str.zfill(3))

# Fall back to municipality harmonic regression planting month
if n_missing_month > 0:
    phenology =  pd.read_csv(phenology_f)
    phenology['muncode'] =  (phenology['CVE_ENT'].astype(str).str.zfill(2) +
                              phenology['CVE_MUN'].astype(str).str.zfill(3))
    phen_map =  phenology.set_index('muncode')['planting_month'].to_dict()

    mask =  plots['planting_month'].isna()
    plots.loc[mask, 'planting_month'] =  plots.loc[mask, 'muncode'].map(phen_map)

    n_still_missing =  plots['planting_month'].isna().sum()
    print(f"After phenology fallback, still missing: {n_still_missing}")

    # Fill any remaining with national median (June = 6)
    if n_still_missing > 0:
        plots['planting_month'] =  plots['planting_month'].fillna(6)

plots['planting_month'] =  plots['planting_month'].astype(int)
print(f"\nPlanting month distribution:")
print(plots['planting_month'].value_counts().sort_index())

Plots missing planting month: 8
After phenology fallback, still missing: 3

Planting month distribution:
planting_month
1      165
2       85
3      963
4     1866
5     3238
6     7515
7     3130
8      518
9       77
10     142
11     546
12     337
Name: count, dtype: int64


#### Build output columns and save

In [8]:
# State code from muncode
plots['state'] =  plots['muncode'].str[:2]

# Convert geometry to GeoJSON string
plots['coords'] =  plots['geometry'].apply(lambda g: to_geojson(force_2d(g)))

# Select output columns
out_cols =  ['plot_id', 'adcid', 'muncode', 'state', 'planting_month', 'area_ha', 'coords']
out =  plots[out_cols].copy()

# Ensure plot_id is string
out['plot_id'] =  out['plot_id'].astype(str)

out.to_csv(out_f, index=False)
print(f"Output written: {out_f}")
print(f"  Rows: {len(out):,}")
print(f"  Unique plots: {out['plot_id'].nunique():,}")
print(f"  States: {out['state'].nunique()}")
print(f"  Municipalities: {out['muncode'].nunique()}")
print(f"  ADCs: {out['adcid'].nunique()}")

Output written: /home/jsayre/Dropbox/Projects/Maize_prediction/Data/CIMMYT/cimmyt_plot_geometries_for_ee.csv
  Rows: 18,582
  Unique plots: 18,582
  States: 29
  Municipalities: 926
  ADCs: 5745
